In [0]:
%run ./01-config

In [0]:
class HistoryLoader():
    def __init__(self, env=None):
        Conf = Config()
        self.test_data_dir = Conf.test_data_dir
        self.catalog = Conf.catalog
        self.db_name = Conf.db_name

    def load_date_lookup(self):
        df = (spark.read
              .format("json")
              .load(f"{self.test_data_dir}/6-date-lookup.json"))

        df = df.selectExpr(
            "CAST(date AS DATE) AS date",
            "CAST(week AS INT) AS week",
            "CAST(year AS INT) AS year",
            "CAST(month AS INT) AS month",
            "CAST(dayofweek AS INT) AS dayofweek",
            "CAST(dayofmonth AS INT) AS dayofmonth",
            "CAST(dayofyear AS INT) AS dayofyear",
            "week_part"
        )

        df.write
        .mode("overwrite")
        .saveAsTable(f"{self.catalog}.{self.db_name}.date_lookup")
        print("Date lookup loaded.")

    def load_history(self):
        self.load_date_lookup()

    def validate(self):
        actual_count = spark.table(f"{self.catalog}.{self.db_name}.date_lookup").count()
        assert actual_count == 365, (f"Expected 365 records, found {actual_count}")
        print("Date lookup validation successful.")